# Stage 2 extensions (GPU): np1, fs1, vLLM, and contracts with pre-check / re-ask

Runs `eval/runner_stage2_ext.py` (a wrapper around the unchanged `eval/runner_stage2.py`). Every extension is off
unless set below; the system name gets the tags (`proposed_pc1rq1_np1_k3__gen_3b`, ...), so nothing mixes with the
existing Stage 2 systems.

- **np1** `NO_PROGRESS_K` (default 3; 0 = off), **fs1** `FEASIBILITY`, **vllm** `BACKEND = "vllm"`.
- **pc1 / rq1 / pc1rq1** (contracts): set `PARSES` to an extension parse folder's `parses.jsonl`:
  `results/stage1/extensions/<parser>_pc1/parses.jsonl` (pre-check, already computed offline and in the repo), or the
  `_rq1` / `_pc1rq1` folders produced by `reask_kaggle.ipynb` (copy them into the repo first, or attach them as a
  Kaggle dataset and give the path).

After the run: download `stage2_ext_bundle.zip`, unzip it at the project root, then
`.venv/bin/python eval/stage2_metrics.py results/stage2 test`.

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
STAGE = "test"
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"; GENERATOR_4BIT = False
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"; AGENT_4BIT = True
NO_PROGRESS_K = 3              # np1: 0 = off
FEASIBILITY = False            # fs1
BACKEND = "hf"                 # "hf" | "vllm" | "pipeline_check" (NOT results)
# (preset, parses.jsonl or None). Contract extensions: point PARSES at an extension parse folder.
RUNS = [
    ("proposed", "results/stage1/extensions/intent_prompted_7b_p1v3_pc1/parses.jsonl"),
    ("gold_contract", None),
]
SEEDS = [0]; TEMPERATURE = 0.0; LIMIT = None
OUT = "/kaggle/working/results/stage2"
CACHE = "/kaggle/working/llm_cache"
GEN_TAG = GENERATOR_MODEL.split("/")[-1].replace("Qwen2.5-", "").replace("-Instruct", "").lower().replace(".", "_")
NEEDED = ["eval/runner_stage2_ext.py", "eval/runner_stage2.py", "src/igt/agents/repair_agent_ext.py"] + \
         [p for _, p in RUNS if p]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

for preset, parses in RUNS:
    tag = f"__{os.path.basename(os.path.dirname(parses))}" if parses else ""
    system = f"{preset}__gen_{GEN_TAG}{tag}"
    cmd = ["python", "eval/runner_stage2_ext.py", "--preset", preset, "--system", system, "--backend", BACKEND,
           "--generator-model", GENERATOR_MODEL, "--agent-model", AGENT_MODEL, "--splits", STAGE,
           "--seeds", *map(str, SEEDS), "--temperature", str(TEMPERATURE), "--cache-dir", CACHE, "--out", OUT,
           "--stop-after-no-progress", str(NO_PROGRESS_K)]
    cmd += ["--generator-4bit"] if GENERATOR_4BIT else []
    cmd += ["--agent-4bit"] if AGENT_4BIT else []
    cmd += ["--feasibility-check"] if FEASIBILITY else []
    cmd += ["--parses", parses] if parses else []
    cmd += ["--limit", str(LIMIT)] if LIMIT else []
    print(f"== {system}: {sh(cmd, f'{OUT}/logs/{system}_{STAGE}.log'):.1f} min")

In [ ]:
# ---------------- Summary ----------------
p = subprocess.run(["python", "eval/stage2_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f"{OUT}/{d}") and d != "logs":
        open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)

In [ ]:
# ---------------- ONE zip with everything needed on the laptop (unzip at the project root) ----------------
BUNDLE = f"{WORK}/stage2_ext_bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["/kaggle/working/results/stage2"]:
        for dp, _, fs in os.walk(base):
            for f in fs:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, WORK))
    z.writestr("stage2_ext_bundle_info.json", json.dumps(dict(commit=COMMIT, created=datetime.datetime.now().isoformat(timespec="seconds")), indent=2))
print("download:", BUNDLE, f"({os.path.getsize(BUNDLE) / 1e6:.1f} MB)")